# သင်ခန်းစာ ၀၁ - AI Agent မိတ်ဆက်ခြင်း

**AI Agents for Beginners** သင်တန်း၏ ပထမဆုံး သင်ခန်းစာသို့ ကြိုဆိုပါတယ်!

**AI agent** ဆိုတာက ကြီးမားတဲ့စာလုံးအနေအထား မော်ဒယ် (LLM) ကို အသုံးပြုပြီး သူ့ရဲ့ ရည်ရွယ်ချက်အတွက် အရေးကြီးသော အရာများကို ပြုလုပ်နိုင်တဲ့ အစိတ်အပိုင်းတစ်ခုဖြစ်ပြီး၊ အပြင်ဘက်ကမ္ဘာမှာ *အကြောင်းပြုလုပ်ဆောင်* နိုင်ပါတယ် — APIများကို ခေါ်ဆိုခြင်း, ဒေတာပြကွက်များကို မေးမြန်းခြင်း, ဒါမှမဟုတ် ကုဒ်များကို လုပ်ဆောင်ခြင်း စတဲ့အရာတွေဖြစ်ပါတယ်။

ဒီ notebookမှာ သင် သင့်ရဲ့ ပထမဆုံး agent ဖြစ်တဲ့ **ခရီးသွားအေးဂျင့်** ကို တည်ဆောက်မှာဖြစ်ပြီး အပျော်အပါးနေရာများအတွက် အကြံပြုချက်ပေးပါလိမ့်မယ်။ လမ်းကြောင်းအတွင်း သင် ဘာတွေသင်ယူမှာလဲဆိုတော့-

1. **Microsoft Agent Framework** ဖြင့် Azure AI Foundry Agent Service နှင့် ချိတ်ဆက်ခြင်း။
2. Agent ကို နည်းပညာတစ်ခုဖြစ်တဲ့ **ကိရိယာ (tool)** — သူခေါ်နိုင်တဲ့ ရိုးရှင်းတဲ့ Python function တစ်ခု ပေးခြင်း။
3. Agent ကို ပြေးဆောင်ပြီး သူ့ရဲ့ ပြန်လည်တုံ့ပြန်မှုကို စစ်ဆေးကြည့်ခြင်း။
4. Agent ရဲ့ ပြန်လည်တုံ့ပြန်မှုကို token-by-token ဖြင့် ဆင့်ကဲထွက်ကြည့်ခြင်း။


## Setup

ဤ notebook ကို မပြေးခင်မှာ သေချာစေရန် ရှိရမည်မှာ-

1. **chat model တစ်ခု ဖြန့်ထားထားသော Azure AI Foundry project တစ်ခု** (ဥပမာ `gpt-4o-mini`)။
2. **Azure CLI ဖြင့် login ရှိရန်** — terminal မှာ `az login` ကို ပြေးရန်။
3. **လိုအပ်သော environment variables များကို သတ်မှတ်ရန်-**
   - `AZURE_AI_PROJECT_ENDPOINT` — သင့် Azure AI Foundry project endpoint။
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — သင့် ဖြန့်ထားသော model အမည်။

အောက်တွင် Python package များကို install ပြုလုပ်ပေးမည့် cell ဖြစ်သည်။


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Creating Your First Agent

Agent တစ်ခုအတွက် လိုအပ်သည်မှာ -

- **ညွှန်ကြားချက်များ** - အဲဒါက *သူက ဒေါ်ဘာလဲ* နဲ့ *ဘယ်လိုပြုမူရမလဲ* (system prompt) ကို ပြောပြပါတယ်။
- **ကိရိယာများ** — Python function များကို `@tool` နဲ့ အလှဆင်ထားပြီး agent က အသုံးပြုပြီး အချက်အလက် ရယူရန် သို့မဟုတ် လုပ်ဆောင်ချက်များ ပြုလုပ်ရန် ခေါ်နိုင်ပါတယ်။

အောက်မှာတော့ လူကြိုက်များတဲ့ ခရီးသွားနေရာများ စာရင်း တစ်ခု ပြန်ပေးတဲ့ ရိုးရှင်းတဲ့ ကိရိယာတစ်ခု ထားထားပါတယ်။ အသုံးပြုသူ က ခရီးသွားကမ်းလှမ်းချက် စုံစမ်းလာတဲ့အခါ agent က ဒီကိရိယာကို အသုံးပြုမှာ ဖြစ်ပါတယ်။


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Responses

အတွေ့အကြုံပိုမိုအပြည့်အဝရှိစေရန် အေဂျင့်၏တုံ့ပြန်ချက်ကို **စီးဆင်းမှု** ဖြင့် ဖော်ပြနိုင်သည်။ ပုံမှန်အဖြေပြန်သည့်အစား၊ အေဂျင့်သည် ဖန်တီးသောစာသားပိုင်းများကို တစိတ်တပိုင်းစီထုတ်ပေးသည်။ ၎င်းသည် အချိန်နှင့်အညီ ပွောဆိုနေပြသလိုခြင်းကို လိုချင်သော စကားပြောမျက်နှာပြင်များတွင် အထူးအသုံးဝင်သည်။


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## စာအကျဉ်း

ဤသင်ခန်းစာတွင် သင် သင်ယူခဲ့သည်မှာ -

- `AzureAIProjectAgentProvider` မှတဆင့် Azure AI Foundry Agent Service နှင့် ချိတ်ဆက်သည့် **provider တစ်ခု ဖန်တီးခြင်း**။
- ကိုယ်ပိုင် Python function များကို agent က ခေါ်နိုင်ရန် `@tool` decorator ဖြင့် **သုံးစွဲနိုင်သော ကိရိယာ တစ်ခု သတ်မှတ်ခြင်း**။
- အသုံးပြုသူ စာတိုက်နှင့်အတူ **agent ကို ပြေးကြည့်ပြီး ၎င်း၏ တုံ့ပြန်မှုကို ပရင့် ထုတ်ပြခြင်း**။
- အချိန်နှင့်တပြေးညီ ထုတ်လွှင့်မှုများအတွက် **တုံ့ပြန်ချက်များကို stream ပြုလုပ်ခြင်း**။

နောက်သင်ခန်းစာတွင် agentic framework များကို ပိုမိုနက်ရှိုင်းစွာတွေ့ရှိပြီး agent များအား ပိုမိုပါဝါကြီးသော ကိရိယာများနှင့် စဉ်ဆက်မပြတ် တွေးခေါ်နိုင်စွမ်းများ ပေးနိုင်ခြင်းကို သင်ကြားမည်ဖြစ်သည်။


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
